# 개방ID `4359300494` 수동 검토

## tl;dr

- `4359300494`는 **용인대학교 스포츠과학대학원**으로 수동 확정할 수 있다.
- 2025년 EDSS 세부 패널의 재적학생 54명·입학생 32명·졸업생 21명·학과 4개가 KEDI 2025 원본과 일치한다.
- 2023년 스포츠웰니스산업대학원에서 스포츠과학대학원으로 명칭이 변경되면서 새 개방ID가 부여된 것으로 판단된다.
- 0101의 입학생·졸업생 값이 0으로 누락되고 지역 문자열도 KEDI와 달라 자동 매칭이 실패했으므로, 수동 KEDI 코드 `53085933` 연결이 필요하다.


## Context & Methods

2025년까지 남아 있으나 KEDI 학교명이 자동 연결되지 않은 개방ID를 학교 수준 EDSS, 세부 패널, KEDI 학교별 교육통계, 용인대학교 공식 자료로 교차검증한다.

### Key Assumptions

- 0101의 0은 다른 EDSS 세부 패널과 모순될 수 있으므로 결측성 0과 실제 0을 구분한다.
- 학과명의 `?`는 원본 인코딩에서 가운데점이 손상된 것으로 보고 공식 표기 `발달재활·특수체육학과`와 비교한다.
- 학교 명칭변경 전후 ID는 합치지 않고 관계만 기록한다.

공식 참고자료:

- https://graduate.yongin.ac.kr/
- https://www.yongin.ac.kr/css/yonginUser/lib/pdf/web/yiu_history%282026-03-01%29.pdf


## Data

- EDSS DuckDB: 고등교육통계 및 대학정보공시 패널
- EDSS–KEDI 신원표 및 행 매칭 증거표
- KEDI 2023~2025 학교별 교육통계 원본 XLSX


In [1]:
import csv
import decimal
import os
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import duckdb

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
database_path = Path(os.environ.get(
    'EDSS_DUCKDB_PATH',
    '/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb',
))
identity_path = repo_root / 'data/processed/edss_0101_kedi_openid_identity_2009_2025.csv'
crosswalk_path = repo_root / 'data/processed/edss_0101_kedi_crosswalk_2009_2025.csv'
kedi_dir = repo_root / 'data/raw/kedi/higher_education_school'
assert database_path.exists() and identity_path.exists() and crosswalk_path.exists() and kedi_dir.exists()
connection = duckdb.connect(str(database_path), read_only=True)
open_id = '4359300494'
duckdb.__version__, database_path.name


('1.4.1', 'edss_all.duckdb')

### 1. 신원표와 자동매칭 실패 원인

In [2]:
with identity_path.open(encoding='utf-8-sig', newline='') as handle:
    identity_row = next(row for row in csv.DictReader(handle) if row['openid'] == open_id)
with crosswalk_path.open(encoding='utf-8-sig', newline='') as handle:
    annual_crosswalk = [row for row in csv.DictReader(handle) if row['openid'] == open_id]
assert identity_row['first_edss_year'] == '2023'
assert identity_row['last_edss_year'] == '2025'
assert identity_row['edss_year_count'] == '3'
assert identity_row['identity_status'] == 'unmatched'
assert {row['direct_match_status'] for row in annual_crosswalk} == {'unmatched'}
identity_row, [{key: row[key] for key in ['year', 'regions', 'campuses', 'direct_match_status']} for row in annual_crosswalk]


({'openid': '4359300494',
  'first_edss_year': '2023',
  'last_edss_year': '2025',
  'edss_year_count': '3',
  'direct_match_year_count': '0',
  'latest_direct_school_name': '',
  'kedi_school_code_2025': '',
  'distinct_normalized_name_count': '0',
  'name_history': '',
  'identity_status': 'unmatched'},
 [{'year': '2023',
   'regions': '경기 용인시 처인구',
   'campuses': '본교',
   'direct_match_status': 'unmatched'},
  {'year': '2024',
   'regions': '경기 용인시 처인구',
   'campuses': '본교',
   'direct_match_status': 'unmatched'},
  {'year': '2025',
   'regions': '경기 용인시 처인구',
   'campuses': '본교(제1캠퍼스)',
   'direct_match_status': 'unmatched'}])

## Results

### 2. 2025년 패널 범위와 비영 값


In [3]:
tables = connection.execute("""
SELECT table_schema, table_name FROM information_schema.columns
WHERE column_name='개방ID'
  AND table_schema IN ('higher_education', 'university_disclosure')
GROUP BY ALL ORDER BY ALL
""").fetchall()
panel_rows = []
for schema_name, table_name in tables:
    row_count = connection.execute(
        f"SELECT COUNT(*) FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    ).fetchone()[0]
    if row_count:
        panel_rows.append((schema_name, table_name, row_count))
assert len(panel_rows) == 24
assert sum(row[2] for row in panel_rows) == 230
panel_rows


[('higher_education', 'panel_0101', 1),
 ('higher_education', 'panel_0104', 1),
 ('higher_education', 'panel_0105', 1),
 ('higher_education', 'panel_0209', 24),
 ('higher_education', 'panel_0212', 20),
 ('higher_education', 'panel_0216', 20),
 ('higher_education', 'panel_0221', 7),
 ('higher_education', 'panel_0231', 8),
 ('higher_education', 'panel_0236', 1),
 ('higher_education', 'panel_0237', 4),
 ('higher_education', 'panel_0245', 2),
 ('higher_education', 'panel_0246', 12),
 ('higher_education', 'panel_0403', 2),
 ('university_disclosure', 'panel_0103', 5),
 ('university_disclosure', 'panel_0306', 3),
 ('university_disclosure', 'panel_0308', 5),
 ('university_disclosure', 'panel_0311', 4),
 ('university_disclosure', 'panel_0313', 1),
 ('university_disclosure', 'panel_0315', 1),
 ('university_disclosure', 'panel_0317', 1),
 ('university_disclosure', 'panel_0502', 3),
 ('university_disclosure', 'panel_0715', 72),
 ('university_disclosure', 'panel_1003', 8),
 ('university_disclosure'

### 3. 학교 수준 0101과 세부 패널의 불일치

In [4]:
school_rows = connection.execute("""
SELECT 조사년도, 지역명, 본분교명, 고등교육학교_재적학생수,
       고등교육학교_재적여학생수, 고등교육학교_입학생수,
       고등교육학교_졸업생수, 고등교육학교_학과수
FROM higher_education.panel_0101 WHERE 개방ID=? ORDER BY 조사년도
""", [open_id]).fetchall()

detailed_totals = []
for year in ('2023', '2024', '2025'):
    entrants = connection.execute("""
        SELECT SUM(CAST(연령별신입생_대학원_입학생수 AS BIGINT))
        FROM higher_education.panel_0209 WHERE 개방ID=? AND 조사년도=?
    """, [open_id, year]).fetchone()[0]
    graduates = connection.execute("""
        SELECT SUM(CAST(연령별졸업생_대학원_졸업생수 AS BIGINT))
        FROM higher_education.panel_0216 WHERE 개방ID=? AND 조사년도=?
    """, [open_id, year]).fetchone()[0]
    detailed_totals.append((year, entrants, graduates))
assert [row[3] for row in school_rows] == ['25', '48', '54']
assert [row[7] for row in school_rows] == ['4', '4', '4']
assert detailed_totals == [('2023', 26, 0), ('2024', 25, 0), ('2025', 32, 21)]
school_rows, detailed_totals


([('2023', '경기 용인시 처인구', '본교', '25', '4', '0', '0', '4'),
  ('2024', '경기 용인시 처인구', '본교', '48', '6', '0', '0', '4'),
  ('2025', '경기 용인시 처인구', '본교(제1캠퍼스)', '54', '12', '0', '0', '4')],
 [('2023', 26, 0), ('2024', 25, 0), ('2025', 32, 21)])

### 4. 학과 구성

In [5]:
department_rows = connection.execute("""
SELECT DISTINCT 조사년도, 학과명, 학과상태명, 주야간계절구분명
FROM university_disclosure.panel_1017 WHERE 개방ID=? ORDER BY 조사년도, 학과명
""", [open_id]).fetchall()
expected_departments = {'골프학과', '발달재활?특수체육학과', '스포츠레저학과', '체육과학과'}
assert {row[1] for row in department_rows} == expected_departments
assert {row[2] for row in department_rows} == {'기존'}
assert {row[3] for row in department_rows} == {'야간'}
department_rows


[('2024', '골프학과', '기존', '야간'),
 ('2024', '발달재활?특수체육학과', '기존', '야간'),
 ('2024', '스포츠레저학과', '기존', '야간'),
 ('2024', '체육과학과', '기존', '야간'),
 ('2025', '골프학과', '기존', '야간'),
 ('2025', '발달재활?특수체육학과', '기존', '야간'),
 ('2025', '스포츠레저학과', '기존', '야간'),
 ('2025', '체육과학과', '기존', '야간')]

### 5. KEDI 원본과 수치 대조

In [6]:
xlsx_ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

def column_index(cell_reference):
    letters = re.match(r'[A-Z]+', cell_reference).group(0)
    result = 0
    for letter in letters:
        result = result * 26 + ord(letter) - 64
    return result - 1

def read_xlsx_rows(path):
    with zipfile.ZipFile(path) as archive:
        shared_root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
        shared = [''.join(node.text or '' for node in item.findall('.//m:t', xlsx_ns))
                  for item in shared_root.findall('m:si', xlsx_ns)]
        sheet_root = ET.fromstring(archive.read('xl/worksheets/sheet1.xml'))
    output = []
    for row in sheet_root.findall('.//m:row', xlsx_ns):
        values = {}
        for cell in row.findall('m:c', xlsx_ns):
            value_node = cell.find('m:v', xlsx_ns)
            if value_node is None:
                continue
            value = shared[int(value_node.text)] if cell.get('t') == 's' else value_node.text
            values[column_index(cell.get('r'))] = value
        output.append([values.get(index, '') for index in range(max(values, default=-1) + 1)])
    return output

kedi_records = []
for year in (2023, 2024, 2025):
    rows = read_xlsx_rows(kedi_dir / f'{year}_kedi_higher_education_school.xlsx')
    header = rows[13]
    record = next(
        {header[index]: row[index] if index < len(row) else '' for index in range(len(header))}
        for row in rows[14:]
        if '용인대학교 스포츠과학대학원' in row
    )
    kedi_records.append((
        str(year), record.get('학교코드', ''), record['학교명'], record['학교상태'],
        record['재적생_전체_계'], record['재적생_전체_여'],
        record['입학자_전체_계'], record['졸업자_전체_계'], record['학과수_전체'],
    ))
assert [row[3] for row in kedi_records] == ['학교명변경', '기존', '기존']
assert [row[4:] for row in kedi_records] == [
    ('25', '4', '26', '0', '4'),
    ('48', '6', '25', '0', '4'),
    ('54', '12', '32', '21', '4'),
]
assert kedi_records[-1][1] == '53085933'
kedi_records


[('2023', '', '용인대학교 스포츠과학대학원', '학교명변경', '25', '4', '26', '0', '4'),
 ('2024', '', '용인대학교 스포츠과학대학원', '기존', '48', '6', '25', '0', '4'),
 ('2025', '53085933', '용인대학교 스포츠과학대학원', '기존', '54', '12', '32', '21', '4')]

### 6. 명칭변경 전후 ID 분리 확인

In [7]:
related_rows = connection.execute("""
SELECT 개방ID, 조사년도, 고등교육학교_재적학생수, 고등교육학교_입학생수,
       고등교육학교_졸업생수, 고등교육학교_학과수
FROM higher_education.panel_0101
WHERE 개방ID IN ('4584404349', '5688246011', '4359300494') AND 조사년도 >= '2022'
ORDER BY 개방ID, 조사년도
""").fetchall()
assert any(row[0] == '5688246011' and row[1] == '2022' and row[2] == '46' for row in related_rows)
assert any(row[0] == '4359300494' and row[1] == '2023' and row[2] == '25' for row in related_rows)
related_rows


[('4359300494', '2023', '25', '0', '0', '4'),
 ('4359300494', '2024', '48', '0', '0', '4'),
 ('4359300494', '2025', '54', '0', '0', '4'),
 ('4584404349', '2022', '1', '0', '0', '1'),
 ('4584404349', '2023', '1', '0', '0', '2'),
 ('4584404349', '2024', '0', '0', '0', '1'),
 ('4584404349', '2025', '1', '0', '0', '1'),
 ('5688246011', '2022', '46', '20', '17', '4'),
 ('5688246011', '2023', '22', '0', '0', '4'),
 ('5688246011', '2024', '6', '0', '0', '4'),
 ('5688246011', '2025', '1', '0', '0', '1')]

## Takeaways

- `4359300494`는 용인대학교 스포츠과학대학원의 **활성·명칭변경 후속 ID**다.
- KEDI 2025 학교코드 `53085933`으로 수동 연결하는 것이 안전하다.
- 구 ID `5688246011`(스포츠웰니스산업대학원), `4584404349`(옛 체육과학대학원)와는 이력 관계만 기록하고 행을 합치지 않는다.
- 자동매칭 실패 원인은 EDSS 지역 `경기 용인시 처인구`와 KEDI 지역 `경기 용인시`의 문자열 차이, 그리고 0101 입학생·졸업생의 누락성 0이다. 세부 패널 0209·0216은 KEDI와 정확히 일치한다.
